In [1]:
import pandas as pd
from pathlib import Path

output_dir = Path("../output")

transactions = pd.read_csv(
    output_dir / "online_retail_clean_backup.csv",
    parse_dates=["InvoiceDate"],
    dtype={
        "Invoice": "string",
        "StockCode": "string"
    }
)

customers = pd.read_csv(
    output_dir / "customer_analytics_final.csv",
    dtype={"Customer ID": "string"}
)

print("Transactions:", transactions.shape)
print("Customer Analytics:", customers.shape)

print("\nCột Transactions:")
print(transactions.columns.tolist())

print("\nCột Customer Analytics:")
print(customers.columns.tolist())

Transactions: (779425, 9)
Customer Analytics: (5878, 6)

Cột Transactions:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'LineTotal']

Cột Customer Analytics:
['Customer ID', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'Inactivity_Risk']


In [2]:
# Chuẩn hóa Customer ID về chuỗi số nguyên
for data in [transactions, customers]:
    data["Customer ID"] = (
        pd.to_numeric(data["Customer ID"], errors="raise")
        .astype("Int64")
        .astype("string")
    )

print("Customer ID trong Transactions:")
print(transactions["Customer ID"].head(5).to_string(index=False))

print("\nCustomer ID trong Customer Analytics:")
print(customers["Customer ID"].head(5).to_string(index=False))

Customer ID trong Transactions:
13085
13085
13085
13085
13085

Customer ID trong Customer Analytics:
12346
12347
12348
12349
12350


In [3]:
transaction_ids = set(transactions["Customer ID"].dropna())
customer_ids = set(customers["Customer ID"].dropna())

print("- KIỂM TRA GOLD DATA -")

print("Số dòng giao dịch:", len(transactions))
print("Số khách hàng:", len(customers))
print("Số hóa đơn:", transactions["Invoice"].nunique())

print("Customer ID thiếu - Transactions:",
      transactions["Customer ID"].isna().sum())

print("Customer ID trùng - Customers:",
      customers["Customer ID"].duplicated().sum())

print("Khách hàng không khớp:",
      len(transaction_ids - customer_ids))

print("Tổng LineTotal:",
      round(transactions["LineTotal"].sum(), 2))

print("Tổng Monetary:",
      round(customers["Monetary"].sum(), 2))

print("Tổng số khách hàng theo Cluster:")
print(customers["Cluster"].value_counts().sort_index().to_string())

print("\nTổng khách hàng theo Risk:")
print(customers["Inactivity_Risk"].value_counts().to_string())

- KIỂM TRA GOLD DATA -
Số dòng giao dịch: 779425
Số khách hàng: 5878
Số hóa đơn: 36969
Customer ID thiếu - Transactions: 0
Customer ID trùng - Customers: 0
Khách hàng không khớp: 0
Tổng LineTotal: 17374804.27
Tổng Monetary: 17374804.27
Tổng số khách hàng theo Cluster:
Cluster
0    1839
1    1675
2    2364

Tổng khách hàng theo Risk:
Inactivity_Risk
Low Risk       2889
High Risk      2401
Medium Risk     588


In [4]:
gold_dir = output_dir / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

transactions.to_csv(
    gold_dir / "transactions_clean.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d %H:%M:%S"
)

customers.to_csv(
    gold_dir / "customer_analytics_final.csv",
    index=False,
    encoding="utf-8-sig"
)

for file in gold_dir.glob("*.csv"):
    print("GOLD FILE:", file.name, "-", round(file.stat().st_size / 1024**2, 2), "MB")

GOLD FILE: customer_analytics_final.csv - 0.18 MB
GOLD FILE: transactions_clean.csv - 71.08 MB


In [5]:
import boto3
from pathlib import Path

bucket_name = "kltn-customer-analytics-2026"

s3 = boto3.client("s3")

gold_files = {
    "gold/transactions_clean.csv":
        Path("../output/gold/transactions_clean.csv"),

    "gold/customer_analytics_final.csv":
        Path("../output/gold/customer_analytics_final.csv")
}

for s3_key, local_path in gold_files.items():
    s3.upload_file(
        str(local_path),
        bucket_name,
        s3_key
    )

    print(f"Uploaded: s3://{bucket_name}/{s3_key}")

Uploaded: s3://kltn-customer-analytics-2026/gold/transactions_clean.csv
Uploaded: s3://kltn-customer-analytics-2026/gold/customer_analytics_final.csv
